# Online Classification

Summary of the best test accuracies from the expand-and-sparsify pipeline
across both datasets.

In [9]:
import numpy as np
import pickle, os
from math import comb
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from tools import load

pkl_rbf = 'data/rbf_svm_results.pkl'

if os.path.exists(pkl_rbf):
    with open(pkl_rbf, 'rb') as f:
        saved_rbf = pickle.load(f)
    table_rbf = saved_rbf['table']
    table_rbf_mix = saved_rbf['table_mix']
    print(f"Loaded RBF SVM results from {pkl_rbf}")
else:
    # --- helpers ---
    def backward_diff_array(y, h, n):
        coeffs = np.array([(-1)**k * comb(n, k) for k in range(n + 1)])
        raw = np.convolve(y, coeffs, mode='valid') / h**n
        return np.concatenate([np.zeros(n), raw])

    def expand_with_derivatives(data, h, max_order):
        if max_order == 0:
            return data.copy()
        derivs = [np.apply_along_axis(lambda c: backward_diff_array(c, h, o),
                                      axis=0, arr=data)
                   for o in range(1, max_order + 1)]
        return np.hstack([data] + derivs)

    n_sensors = 8
    rng_pairs = np.random.default_rng(0)
    all_ordered_pairs = [(i, j) for i in range(n_sensors)
                         for j in range(n_sensors) if i != j]
    ratio_pairs = [all_ordered_pairs[i]
                   for i in rng_pairs.permutation(len(all_ordered_pairs))]
    diff_pairs  = [all_ordered_pairs[i]
                   for i in rng_pairs.permutation(len(all_ordered_pairs))]

    n_repeats = 100

    def build_configs(sensor_data, h):
        x_d1  = expand_with_derivatives(sensor_data, h, 1)
        x_d12 = expand_with_derivatives(sensor_data, h, 2)
        x_d123 = expand_with_derivatives(sensor_data, h, 3)
        r8 = np.column_stack([sensor_data[:, i] / (sensor_data[:, j] + 1e-8)
                               for i, j in ratio_pairs[:8]])
        d8 = np.column_stack([sensor_data[:, i] - sensor_data[:, j]
                               for i, j in diff_pairs[:8]])
        return {
            '∂¹+∂²+∂³':    x_d123,
            '∂¹+∂² + 8R':  np.hstack([x_d12, r8]),
            '∂¹+∂² + 8D':  np.hstack([x_d12, d8]),
            '∂¹ + 8R + 8D': np.hstack([x_d1, r8, d8]),
        }

    def make_labels(times_sec, sequence, sequence_sec):
        labels = np.zeros_like(times_sec)
        for i in range(len(sequence_sec)):
            try:
                flag = (times_sec > sequence_sec[i]) & (times_sec < sequence_sec[i + 1])
            except IndexError:
                flag = (times_sec > sequence_sec[i])
            labels[flag] = int(sequence[i][1])
        mask = labels > 0
        return labels[mask].astype(int) - 1, mask

    def run_rbf_svm(configs, y, n_repeats):
        results = {}
        for name, x_exp in configs.items():
            accs = []
            for seed in range(n_repeats):
                tr_idx, te_idx = train_test_split(
                    np.arange(len(y)), test_size=0.2,
                    random_state=seed, stratify=y)
                clf = SVC(kernel='rbf', C=50, gamma='scale')
                clf.fit(x_exp[tr_idx], y[tr_idx])
                accs.append(clf.score(x_exp[te_idx], y[te_idx]))
            results[name] = (np.mean(accs), np.std(accs))
            print(f"  {name}: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
        return results

    # --- 1_600_20 ---
    print("=== 1_600_20 (RBF SVM) ===")
    sd1, seq1, ts1, ss1 = load('1_600_20', reduced=True)
    h1 = np.median(np.diff(ts1))
    y1, mask1 = make_labels(ts1, seq1, ss1)
    cfgs1 = {k: v[mask1] for k, v in build_configs(sd1, h1).items()}
    table_rbf = run_rbf_svm(cfgs1, y1, n_repeats)

    # --- mix_100_20_1 ---
    print("\n=== mix_100_20_1 (RBF SVM) ===")
    sd2, seq2, ts2, ss2 = load('mix_100_20_1', reduced=True)
    h2 = np.median(np.diff(ts2))
    y2, mask2 = make_labels(ts2, seq2, ss2)
    cfgs2 = {k: v[mask2] for k, v in build_configs(sd2, h2).items()}
    table_rbf_mix = run_rbf_svm(cfgs2, y2, n_repeats)

    # Save
    with open(pkl_rbf, 'wb') as f:
        pickle.dump({'table': table_rbf, 'table_mix': table_rbf_mix}, f)
    print(f"\nSaved to {pkl_rbf}")

Loaded RBF SVM results from data/rbf_svm_results.pkl


In [10]:
import pickle
import numpy as np
from IPython.display import HTML, display

# Load expand & sparsify results
with open('data/expand_sparsify_results.pkl', 'rb') as f:
    res_single = pickle.load(f)
with open('data/expand_sparsify_mix_results.pkl', 'rb') as f:
    res_mix = pickle.load(f)

# Load linear SVM on x_dense results (from R1)
with open('data/mixed_expansion_dense_results.pkl', 'rb') as f:
    res_dense = pickle.load(f)

# Load RBF SVM results
with open('data/rbf_svm_results.pkl', 'rb') as f:
    res_rbf = pickle.load(f)

# Load Hebbian framewise gridsearch results
with open('data/gs_single_framewise.pkl', 'rb') as f:
    gs_single = pickle.load(f)
with open('data/gs_binary_framewise.pkl', 'rb') as f:
    gs_binary = pickle.load(f)

at_single = res_single['acc_table']
at_mix = res_mix['acc_table']
table_dense = res_dense['table']
table_dense_mix = res_dense['table_mix']
config_names = res_single['config_names']
n_configs = len(config_names)

# Extract best Hebbian accuracy per expansion
def best_hebbian(gs, expansion):
    params = gs['params']
    test_acc = gs['results']['test_acc']
    p_hds = np.unique(params['p_hd'])
    ds = np.unique(params['d'])
    best_mean, best_std = -1, 0
    for phd in p_hds:
        for d in ds:
            mask = ((params['expansion'] == expansion) &
                    (np.abs(params['p_hd'] - phd) < 1e-6) &
                    (np.abs(params['d'] - d) < 1e-6))
            if mask.sum() == 0:
                continue
            accs = test_acc[mask]
            m = accs.mean()
            if m > best_mean:
                best_mean, best_std = m, accs.std()
    return best_mean, best_std

# --- Build rows ---
rows_html = []

def add_method_block(method_name, readout, get_acc, first_block=True):
    for i, cfg in enumerate(config_names):
        m1, s1, m2, s2 = get_acc(cfg)
        border = '' if first_block and i == 0 else (
            ' style="border-top:2px solid #888;"' if i == 0 else '')
        method_cell = (
            f'<td rowspan="{n_configs}" style="vertical-align:middle;'
            + ('' if first_block and i == 0 else ' border-top:2px solid #888;')
            + f'">{method_name}</td>'
            if i == 0 else ''
        )
        readout_cell = (
            f'<td rowspan="{n_configs}" style="vertical-align:middle;'
            + ('' if first_block and i == 0 else ' border-top:2px solid #888;')
            + f'">{readout}</td>'
            if i == 0 else ''
        )
        rows_html.append(
            f'<tr>{method_cell}{readout_cell}<td{border}>{cfg}</td>'
            f'<td{border}>{m1:.3f} &plusmn; {s1:.3f}</td>'
            f'<td{border}>{m2:.3f} &plusmn; {s2:.3f}</td></tr>'
        )

# Expand & sparsify — Hebbian binary weights
add_method_block('Expand &amp; sparsify', 'Hebbian &#8211; binary weights',
    lambda cfg: (*best_hebbian(gs_single, cfg), *best_hebbian(gs_binary, cfg)),
    first_block=True)

# Expand & sparsify — Linear classifier
def es_acc(cfg):
    bs = max((v for k, v in at_single.items() if k[0] == cfg), key=lambda x: x[0])
    bm = max((v for k, v in at_mix.items()    if k[0] == cfg), key=lambda x: x[0])
    return (*bs, *bm)

add_method_block('Expand &amp; sparsify', 'Linear classifier', es_acc, first_block=False)

# Linear SVM
add_method_block('Linear SVM', 'NA',
    lambda cfg: (*table_dense[cfg], *table_dense_mix[cfg]),
    first_block=False)

# RBF SVM
add_method_block('RBF SVM', 'NA',
    lambda cfg: (*res_rbf['table'][cfg], *res_rbf['table_mix'][cfg]),
    first_block=False)

html_table = f'''<table style="border-collapse:collapse;">
<thead>
<tr>
  <th rowspan="2" style="text-align:left; vertical-align:bottom;">Method</th>
  <th rowspan="2" style="text-align:left; vertical-align:bottom;">Readout</th>
  <th rowspan="2" style="text-align:left; vertical-align:bottom;">Expansion</th>
  <th colspan="2" style="text-align:center; border-bottom:1px solid #aaa;">Test accuracy</th>
</tr>
<tr>
  <th style="text-align:left;">Single gases (n=600)</th>
  <th style="text-align:left;">Binary mixtures (n=600)</th>
</tr>
</thead>
<tbody>
{chr(10).join(rows_html)}
</tbody>
</table>'''

display(HTML(html_table))

### LaTeX

```latex
\begin{table*}[t]
\centering
\caption{Test accuracy (mean $\pm$ std) for four classification approaches
         on four trace-expansion configurations.}
\label{tab:online-classification}
\begin{tabular}{@{}lll cc@{}}
\toprule
         &                 &                          & \multicolumn{2}{c}{Test accuracy} \\
         \cmidrule(l){4-5}
Method   & Readout        & Expansion                & Single gases ($n{=}600$)
                                                       & Binary mixtures ($n{=}600$) \\
\midrule
Expand \& sparsify
         & Hebbian -- binary weights
         & $\partial^1{+}\partial^2{+}\partial^3$
                                    & $0.930 \pm 0.003$ & $0.541 \pm 0.014$ \\
         &
         & $\partial^1{+}\partial^2$ + 8R
                                    & $0.955 \pm 0.001$ & $0.566 \pm 0.014$ \\
         &
         & $\partial^1{+}\partial^2$ + 8D
                                    & $0.907 \pm 0.005$ & $0.553 \pm 0.030$ \\
         &
         & $\partial^1$ + 8R + 8D   & $0.910 \pm 0.007$ & $0.594 \pm 0.025$ \\
\midrule
Expand \& sparsify
         & Linear classifier
         & $\partial^1{+}\partial^2{+}\partial^3$
                                    & $0.964 \pm 0.003$ & $0.721 \pm 0.008$ \\
         &
         & $\partial^1{+}\partial^2$ + 8R
                                    & $0.967 \pm 0.003$ & $0.777 \pm 0.007$ \\
         &
         & $\partial^1{+}\partial^2$ + 8D
                                    & $0.965 \pm 0.003$ & $0.807 \pm 0.008$ \\
         &
         & $\partial^1$ + 8R + 8D   & $0.963 \pm 0.004$ & $0.788 \pm 0.011$ \\
\midrule
Linear SVM
         & NA
         & $\partial^1{+}\partial^2{+}\partial^3$
                                    & $0.938 \pm 0.052$ & $0.611 \pm 0.074$ \\
         &
         & $\partial^1{+}\partial^2$ + 8R
                                    & $0.938 \pm 0.044$ & $0.622 \pm 0.074$ \\
         &
         & $\partial^1{+}\partial^2$ + 8D
                                    & $0.921 \pm 0.054$ & $0.575 \pm 0.082$ \\
         &
         & $\partial^1$ + 8R + 8D   & $0.915 \pm 0.057$ & $0.562 \pm 0.073$ \\
\midrule
RBF SVM
         & NA
         & $\partial^1{+}\partial^2{+}\partial^3$
                                    & $0.958 \pm 0.004$ & $0.776 \pm 0.008$ \\
         &
         & $\partial^1{+}\partial^2$ + 8R
                                    & $0.958 \pm 0.004$ & $0.776 \pm 0.008$ \\
         &
         & $\partial^1{+}\partial^2$ + 8D
                                    & $0.956 \pm 0.004$ & $0.777 \pm 0.008$ \\
         &
         & $\partial^1$ + 8R + 8D   & $0.951 \pm 0.004$ & $0.773 \pm 0.008$ \\
\bottomrule
\end{tabular}
\end{table*}
```